# Consolidated Overall Reconciliation Tests

End-to-end record-count + uniqueness checks across **all three reconciliation universes**, plus a parquet -> delta sanity check at the top to surface any landing-layer breakage before the reconciliation logic runs.

```
Appeals data — Active appeals (raw_appealcase, CaseType=1)
  raw_CT1 = active + FPA_stg + FTA_stg + UTA_stg + TD_stg + leftover
  per-TargetState gold parity + per-segment stg->gold parity
  dept-519 spec assertion + dept-520 routing consistency

Bails data — BAIL (aria_bails.raw_appeal_cases, CaseType=2)
  raw_CT2 -> dev-code filter -> silver_bail_combined_segmentation_nb_lhnb -> create_bails_json_content
  segmentation count ~= gold count, gold CaseNo uniqueness

Scottish Bails data — SBAIL (Scottish__Bailsfile.csv -> silver_scottish_sbails_funds)
  segmentation count ~= gold count, gold CaseNo uniqueness

Cross-cutting (A + B + C):
  No CaseNo in 2+ of {FPA, FTA, UTA, BAIL, SBAIL, TD}
  No CaseNo in active intersect any archive
  No CaseNo in 2+ TargetStates within active
```

**JOH out of scope** — JOH archives judicial *officers* (keyed by AdjudicatorId, not CaseNo) and has its own notebook `2_JOH_RECONCILIATION_TESTS`.

Everything lands as one run in `test_automation_runs2` / `test_automation_results2` / `test_automation_perfresults2` — single `run_id`, single banner, single CSV/XLSX/HTML. The `test_from_state` column carries the universe label (`active_reconciliation` / `bail_reconciliation` / `sbail_reconciliation` / `overall_cross_cutting`) so downstream queries can filter by universe.

In [0]:
# Notebook-wide config.  Edit a table name here and re-run; the bad name will surface as NO_DATA.
# state_under_test is rebound per-universe by _set_scope() before each section's tests fire.
state_under_test = "overall_reconciliation"
run_tag_default  = "overall_reconciliation"

dbutils.widgets.text("run_tag", run_tag_default)
run_tag = dbutils.widgets.get("run_tag") or run_tag_default

# ---- Appeals data — Active appeals
ACTIVE_SEG_TBL  = "hive_metastore.ariadm_active_appeals.stg_segmentation_states"
ACTIVE_RAW_TBL  = "hive_metastore.ariadm_active_appeals.raw_appealcase"
ACTIVE_FL_TBL   = "hive_metastore.ariadm_active_appeals.raw_filelocation"

ACTIVE_STATE_TO_GOLD = {
    "appealSubmitted": "hive_metastore.appealsubmitted_gold.stg_valid_appeal_submitted_records",
    "paymentPending": "hive_metastore.paymentPending_gold.stg_valid_paymentPending_records",
    "awaitingRespondentEvidence(a)": "hive_metastore.awaitingrespondentevidencea_gold.stg_valid_awaiting_respondent_evidence_a_records",
    "awaitingRespondentEvidence(b)": "hive_metastore.awaitingrespondentevidenceb_gold.stg_valid_awaiting_respondent_evidence_b_records",
    "caseUnderReview": "hive_metastore.caseunderreview_gold.stg_valid_case_under_review_records",
    "decided(a)": "hive_metastore.decideda_gold.stg_valid_decided_a_records",
    "decided(b)": "hive_metastore.decidedb_gold.stg_valid_decided_b_records",
    "decision": "hive_metastore.decision_gold.stg_valid_decision_records",
    "ended": "hive_metastore.ended_gold.stg_valid_ended_records",
    "ftpaDecided": "hive_metastore.ftpadecided_gold.stg_valid_ftpadecided_records",
    "ftpaSubmitted(a)": "hive_metastore.ftpasubmitteda_gold.stg_valid_ftpa_submitted_a_records",
    "ftpaSubmitted(b)": "hive_metastore.ftpasubmittedb_gold.stg_valid_ftpa_submitted_b_records",
    "listing": "hive_metastore.listing_gold.stg_valid_listing_records",
    "prepareForHearing": "hive_metastore.prepareforhearing_gold.stg_valid_prepare_for_hearing_records",
    "reasonsForAppealSubmitted": "hive_metastore.reasonsforappealsubmitted_gold.stg_valid_reasons_for_appeal_submitted_records",
    "remitted": "hive_metastore.remitted_gold.stg_valid_remitted_records"
}

# Per-state DQ quarantine tables. stg_invalid_<state>_quarantine_records — records dropped by
# @dlt.expect_all on the gold step. segmentation count = valid count + invalid count per state.
ACTIVE_STATE_TO_INVALID = {
    "appealSubmitted": "hive_metastore.appealsubmitted_gold.stg_invalid_appeal_submitted_quarantine_records",
    "paymentPending": "hive_metastore.paymentPending_gold.stg_invalid_paymentPending_quarantine_records",
    "awaitingRespondentEvidence(a)": "hive_metastore.awaitingrespondentevidencea_gold.stg_invalid_awaiting_respondent_evidence_a_quarantine_records",
    "awaitingRespondentEvidence(b)": "hive_metastore.awaitingrespondentevidenceb_gold.stg_invalid_awaiting_respondent_evidence_b_quarantine_records",
    "caseUnderReview": "hive_metastore.caseunderreview_gold.stg_invalid_case_under_review_quarantine_records",
    "decided(a)": "hive_metastore.decideda_gold.stg_invalid_decided_a_quarantine_records",
    "decided(b)": "hive_metastore.decidedb_gold.stg_invalid_decided_b_quarantine_records",
    "decision": "hive_metastore.decision_gold.stg_invalid_decision_quarantine_records",
    "ended": "hive_metastore.ended_gold.stg_invalid_ended_quarantine_records",
    "ftpaDecided": "hive_metastore.ftpadecided_gold.stg_invalid_ftpadecided_quarantine_records",
    "ftpaSubmitted(a)": "hive_metastore.ftpasubmitteda_gold.stg_invalid_ftpa_submitted_a_quarantine_records",
    "ftpaSubmitted(b)": "hive_metastore.ftpasubmittedb_gold.stg_invalid_ftpa_submitted_b_quarantine_records",
    "listing": "hive_metastore.listing_gold.stg_invalid_listing_quarantine_records",
    "prepareForHearing": "hive_metastore.prepareforhearing_gold.stg_invalid_prepare_for_hearing_quarantine_records",
    "reasonsForAppealSubmitted": "hive_metastore.reasonsforappealsubmitted_gold.stg_invalid_reasons_for_appeal_submitted_quarantine_records",
    "remitted": "hive_metastore.remitted_gold.stg_invalid_remitted_quarantine_records"
}

# Archive segments visible to the active universe.
# (label, schema, stg_table, gold_table) — stg_table=None for BAIL/SBAIL (handled by the Bails data / Scottish Bails data sections).
ARCHIVE_SEGMENTS = [('FPA', 'ariadm_arm_fpa', 'stg_filepreservedcases_filtered', 'gold_appeals_with_json'), ('FTA', 'ariadm_arm_fta', 'stg_appeals_filtered', 'gold_appeals_with_json'), ('UTA', 'ariadm_arm_uta', 'stg_appeals_filtered', 'gold_appeals_with_json'), ('BAIL', 'aria_bails', None, 'create_bails_json_content'), ('SBAIL', 'aria_s_bails', None, 'create_sbail_json_content'), ('TD', 'ariadm_arm_td', 'stg_td_filtered', 'gold_td_iris_with_json')]
EQUATION_SEGMENTS = ['FPA', 'FTA', 'UTA', 'TD']
ARCHIVE_CASENO_SEGMENTS = ['FPA', 'FTA', 'UTA', 'BAIL', 'SBAIL', 'TD']

# ---- Bails data — BAIL
BAIL_RAW_TBL  = "hive_metastore.aria_bails.raw_appeal_cases"
BAIL_FL_TBL   = "hive_metastore.aria_bails.raw_file_location"
BAIL_SEG_TBL  = "hive_metastore.aria_bails.silver_bail_combined_segmentation_nb_lhnb"
BAIL_GOLD_TBL = "hive_metastore.aria_bails.create_bails_json_content"

# ---- Scottish Bails data — SBAIL
SBAIL_SEG_TBL  = "hive_metastore.aria_s_bails.silver_scottish_sbails_funds"
SBAIL_GOLD_TBL = "hive_metastore.aria_s_bails.create_sbail_json_content"

# ---- Parquet -> Delta checks (universe_label, source_folder, raw_delta_fqn)
PARQUET_DELTA_CHECKS = [('A', 'AppealCase', 'hive_metastore.ariadm_active_appeals.raw_appealcase'), ('A', 'FileLocation', 'hive_metastore.ariadm_active_appeals.raw_filelocation'), ('A', 'CaseAppellant', 'hive_metastore.ariadm_active_appeals.raw_caseappellant'), ('A', 'Appellant', 'hive_metastore.ariadm_active_appeals.raw_appellant'), ('B', 'AppealCase', 'hive_metastore.aria_bails.raw_appeal_cases'), ('B', 'FileLocation', 'hive_metastore.aria_bails.raw_file_location')]

In [0]:
import os
import sys
import time as perf_time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.utils import AnalysisException

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Overall_Reconciliation_Tests"
os.makedirs(test_results_path, exist_ok=True)

# Force a fresh import of the support modules (notebook reruns otherwise keep stale copies).
for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv,
    count_by_status,
    create_run, create_results, create_perf_results,
)

print(f"User: {run_user}")
print(f"state_under_test: {state_under_test} (will rebind per universe)")
print(f"Results folder root: {test_results_path}")

In [0]:
#######################
# Spark Config — SP creds for ADLS-backed tables.
# Storage account list is the UNION across A + B + C — currently identical for all three so
# one cell suffices, but kept explicit so adding a universe-specific account is a one-line edit.
#######################
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
first_row = config.first()
env = first_row["env"].strip().lower()
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

storage_accounts = [
    f"ingest{lz_key}curated{env}",   # bronze/silver/gold delta
    f"ingest{lz_key}xcutting{env}",  # cross-cutting reference data
    f"ingest{lz_key}raw{env}",       # raw zone (parquet -> delta landing)
    f"ingest{lz_key}landing{env}",   # SQLServer parquet drops
    f"ingest{lz_key}external{env}",  # external (e.g. Scottish__Bailsfile.csv)
]
for storage in storage_accounts:
    spark.conf.set(f"fs.azure.account.auth.type.{storage}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage}.dfs.core.windows.net",
                   "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage}.dfs.core.windows.net",
                   f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Landing root (used by the parquet -> delta sanity check below)
landing_path = f"abfss://landing@ingest{lz_key}landing{env}.dfs.core.windows.net/SQLServer/Sales/IRIS/dbo"

print(f"env={env}  lz_key={lz_key}")
print(f"landing_path={landing_path}")

In [0]:
# ---- Shared state.
# counts is keyed per universe so concurrent universes never collide:
#   counts["A"]  Active appeals
#   counts["B"]  BAIL
#   counts["C"]  SBAIL
#   counts["X"]  cross-cutting (overlaps)
# Per-universe sub-dicts are populated by the matching gather cell.
counts = {"A": {}, "B": {}, "C": {}, "X": {}}

all_test_results = []
perf_timings     = []
run_start_datetime = datetime.utcnow()

# DataFrame cache so cross-cutting uniqueness in section X can reuse the distinct-CaseNo
# DataFrames that Bails data / Scottish Bails data gather already loaded — saves a second .distinct().count().
_df_cache = {}

def _set_scope(label):
    """Rebind state_under_test so every TestResult and perf row tagged after this call
    carries `label` in its test_from_state column."""
    global state_under_test
    state_under_test = label
    print(f"  >>> scope: {label}")

def _try_load(fq_table):
    """Load a fully-qualified Delta table. Returns None on any failure (test reports NO_DATA)."""
    try:
        return spark.table(fq_table)
    except Exception as e:
        print(f"  ! cannot load {fq_table}: {str(e)[:160]}")
        return None

def _load_distinct_caseno(fq_table, cache_key=None):
    """Load CaseNo-distinct DF for a gold/silver table, caching by table name (or explicit key).
    Returns None if table not loadable or has no CaseNo column."""
    key = cache_key or fq_table
    if key in _df_cache:
        return _df_cache[key]
    df = _try_load(fq_table)
    if df is None or "CaseNo" not in df.columns:
        _df_cache[key] = None
        return None
    distinct_df = df.select("CaseNo").distinct()
    distinct_df.cache()
    _df_cache[key] = distinct_df
    return distinct_df

def _run_timed(test_name, fn, *args, **kwargs):
    """Run a test function and capture (elapsed, result_count) for the perf table.
    test_from_state on the perf row uses whatever scope is bound at run time."""
    before = len(all_test_results)
    t0 = perf_time.perf_counter()
    fn(*args, **kwargs)
    elapsed = perf_time.perf_counter() - t0
    perf_timings.append({
        "test_from_state": state_under_test,
        "test_name": test_name,
        "elapsed_seconds": elapsed,
        "result_count": len(all_test_results) - before,
    })

def _add(test_name, field, status, message="", description=""):
    """Append a TestResult tagged with the currently-bound scope."""
    all_test_results.append(TestResult(
        test_name=test_name, test_field=field, test_from_state=state_under_test,
        status=status, message=message, description=description,
    ))

def _fmt_n(n):
    return f"{n:>10,}" if isinstance(n, int) else f"{'?':>10}"

## Pre-check: Parquet -> Delta sanity sweep

Before any reconciliation runs, we verify every parquet folder we will indirectly depend on has at least one parquet file present in `landing_path` and that the corresponding `raw_*` Delta table loads. A missing landing folder or a broken raw table makes all downstream reconciliation noise — this section surfaces it up front, scoped as `overall_parquet_delta_check`.

In [0]:
_set_scope("overall_parquet_delta_check")

# Mirror the dev-code pattern from BRONZE_ACTIVE_APPEALS.ipynb:
#   - deep_ls() walks the folder recursively (parquet drops sit under timestamped subdirs)
#   - extract _<timestamp>.parquet, pick max, read THAT file's row count
# That row count is what the dev pipeline ingests, so the right thing to compare against raw_ delta.
from pyspark.sql.functions import regexp_extract as _re_extract, max as _spark_max
from pyspark.sql.functions import col as _col

def _deep_ls(path, depth=0, max_depth=10):
    out = set()
    if depth > max_depth:
        return list(out)
    try:
        for child in dbutils.fs.ls(path):
            if child.path.endswith(".parquet"):
                out.add(child.path.strip())
            if child.isDir:
                out.update(_deep_ls(child.path, depth + 1, max_depth))
    except Exception:
        pass
    return list(out)

def _latest_parquet_row_count(folder_full_path):
    """Mirror read_latest_parquet: deep_ls -> max timestamp -> read that file -> count.
    Returns (latest_file_path, row_count, all_files_count) or (None, None, 0) if no parquet found."""
    all_files = _deep_ls(folder_full_path)
    if not all_files:
        return (None, None, 0)
    file_df = spark.createDataFrame([(f,) for f in all_files], ["file_path"])
    file_df = file_df.withColumn("ts", _re_extract("file_path", r"_(\d+)\.parquet$", 1).cast("long"))
    max_ts = file_df.agg(_spark_max("ts")).collect()[0][0]
    if max_ts is None:
        # parquets exist but don't follow the _<timestamp>.parquet pattern — take any one
        latest = all_files[0]
    else:
        latest = file_df.filter(_col("ts") == max_ts).first()["file_path"]
    try:
        n = spark.read.parquet(latest).count()
    except Exception:
        n = None
    return (latest, n, len(all_files))

def test_parquet_delta_sanity():
    desc = "Per consumed table: latest parquet file row count (deep_ls + max timestamp) should equal raw_* delta count"
    for universe, folder, raw_fqn in PARQUET_DELTA_CHECKS:
        field = f"{universe}:{folder}"

        # 1) Find and count the latest parquet drop
        full_folder = f"{landing_path}/{folder}/full/"
        latest_path, parquet_count, n_files = _latest_parquet_row_count(full_folder)

        if parquet_count is None:
            _add("test_parquet_delta_sanity", f"landing:{field}", "FAIL",
                 f"no parquet found via deep_ls in {folder}/full/ ({n_files} files seen total)", desc)
            # Still try the delta side
            df = _try_load(raw_fqn)
            if df is None:
                _add("test_parquet_delta_sanity", f"parity:{field}", "FAIL",
                     f"raw delta also not loadable: {raw_fqn}", desc)
            continue

        _add("test_parquet_delta_sanity", f"landing:{field}", "PASS",
             f"latest parquet has {parquet_count} rows ({n_files} files in tree) — {latest_path.rsplit('/', 1)[-1]}", desc)

        # 2) Raw Delta table loads + count matches the latest parquet
        df = _try_load(raw_fqn)
        if df is None:
            _add("test_parquet_delta_sanity", f"parity:{field}", "FAIL",
                 f"raw delta table not loadable: {raw_fqn}", desc)
            continue
        try:
            delta_count = df.count()
        except Exception as e:
            _add("test_parquet_delta_sanity", f"parity:{field}", "FAIL",
                 f"{raw_fqn} loaded but count failed: {str(e)[:120]}", desc)
            continue

        diff = parquet_count - delta_count
        if diff == 0:
            _add("test_parquet_delta_sanity", f"parity:{field}", "PASS",
                 f"parquet={parquet_count} delta={delta_count} match", desc)
        else:
            _add("test_parquet_delta_sanity", f"parity:{field}", "FAIL",
                 f"parquet={parquet_count} delta={delta_count} diff={diff} — raw load drifted from latest parquet", desc)

_run_timed("test_parquet_delta_sanity", test_parquet_delta_sanity)

## Appeals data — Active appeals (raw_appealcase, CaseType=1)

Total reconciliation equation:

```
raw_appealcase WHERE CaseType=1
  = active                  (stg_segmentation_states with non-null TargetState)
  + FPA_stg                 (file preserved — DeptId=520 bucket — pushes to ARM)
  + FTA_stg                 (first-tier appeals master union — pushes to ARM)
  + UTA_stg                 (upper-tribunal master union — pushes to ARM)
  + TD_stg                  (tribunal decisions incl. 519 destroyed-file cases — pushes to ARM)
  + leftover
```

Plus per-TargetState gold parity, per-segment stg -> gold parity, dept-519 spec assertion, dept-520 routing consistency, within-active uniqueness.

All results in this section are tagged `test_from_state = active_reconciliation`.

In [0]:
_set_scope("active_reconciliation")

def test_gather_counts_A():
    desc = "A: raw / active / per-segment stg+gold / dept-519 / dept-520 counts"
    A = counts["A"]

    # --- Raw baseline + filelocation (cache the joined CT=1 view; reused by 519/520 isolation later)
    raw_df = _try_load(ACTIVE_RAW_TBL)
    fl_df  = _try_load(ACTIVE_FL_TBL)
    if raw_df is None:
        _add("test_gather_counts_A", "raw_appealcase", "NO_DATA", f"could not load {ACTIVE_RAW_TBL}", desc)
        A["raw_CT1"] = None
    else:
        # Use distinct CaseNo so RHS terms (which are also distinct CaseNo) can balance
        A["raw_CT1"] = raw_df.filter(col("CaseType") == 1).select("CaseNo").distinct().count()
        _add("test_gather_counts_A", "raw_appealcase", "PASS",
             f"{A['raw_CT1']} distinct CaseType=1 CaseNos in raw_appealcase", desc)

    # Pre-compute the CT=1 × filelocation join once; subsequent dept tests reuse the cached distinct sets.
    if raw_df is not None and fl_df is not None:
        ac_fl_519 = (raw_df.alias("ac")
                     .join(fl_df.alias("fl"), col("ac.CaseNo") == col("fl.CaseNo"), "left")
                     .filter((col("ac.CaseType") == 1) & (col("fl.DeptId") == 519))
                     .select(col("ac.CaseNo").alias("CaseNo")).distinct())
        ac_fl_519.cache()
        _df_cache["A:dept519_caseno"] = ac_fl_519
        A["dropped_519"] = ac_fl_519.count()
        _add("test_gather_counts_A", "dropped_519", "PASS",
             f"{A['dropped_519']} distinct CaseNos with DeptId=519 — should be left behind per spec", desc)

        ac_fl_520 = (raw_df.alias("ac")
                     .join(fl_df.alias("fl"), col("ac.CaseNo") == col("fl.CaseNo"), "left")
                     .filter((col("ac.CaseType") == 1) & (col("fl.DeptId") == 520))
                     .select(col("ac.CaseNo").alias("CaseNo")).distinct())
        ac_fl_520.cache()
        _df_cache["A:dept520_caseno"] = ac_fl_520
        A["dept_520"] = ac_fl_520.count()
        _add("test_gather_counts_A", "dept_520", "PASS",
             f"{A['dept_520']} distinct CaseNos with DeptId=520 — should land in FPA", desc)
    else:
        A["dropped_519"] = None
        A["dept_520"] = None

    # --- Active segmentation (cached for downstream cross-cutting)
    seg_df = _try_load(ACTIVE_SEG_TBL)
    if seg_df is None:
        A["active_stg"] = None
        _add("test_gather_counts_A", "active_stg", "NO_DATA", f"could not load {ACTIVE_SEG_TBL}", desc)
    else:
        active_cases = seg_df.filter(col("TargetState").isNotNull()).select("CaseNo").distinct()
        active_cases.cache()
        _df_cache["A:active_distinct"] = active_cases
        A["active_stg"] = active_cases.count()
        _add("test_gather_counts_A", "active_stg", "PASS",
             f"{A['active_stg']} distinct CaseNos with non-null TargetState in stg_segmentation_states", desc)

    # --- Archive: per segment, capture stg + gold counts as DISTINCT CaseNo so the
    # universe equation (raw = active + sum of segments) balances on a like-for-like basis.
    # Some stg tables (e.g. stg_appeals_filtered) are a union of sub-stages and CAN have
    # duplicate CaseNos — using row-count there caused the previous -3,416 negative leftover.
    A["archive_by_segment"] = {}
    for label, schema, stg_tbl, gold_tbl in ARCHIVE_SEGMENTS:
        seg_info = {"stg": None, "gold": None, "stg_rows": None, "gold_rows": None}

        if stg_tbl is not None:
            df = _try_load(f"hive_metastore.{schema}.{stg_tbl}")
            if df is not None:
                seg_info["stg_rows"] = df.count()
                if "CaseNo" in df.columns:
                    seg_info["stg"] = df.select("CaseNo").distinct().count()
                else:
                    # No CaseNo column — fall back to row count, but flag the limitation
                    seg_info["stg"] = seg_info["stg_rows"]
                dup_note = ""
                if seg_info["stg_rows"] != seg_info["stg"]:
                    dup_note = f" (rows={seg_info['stg_rows']} — {seg_info['stg_rows'] - seg_info['stg']} duplicate CaseNo rows)"
                _add("test_gather_counts_A", f"{label}_stg", "PASS",
                     f"{seg_info['stg']} distinct CaseNos in {schema}.{stg_tbl}{dup_note}", desc)
            else:
                _add("test_gather_counts_A", f"{label}_stg", "NO_DATA",
                     f"could not load {schema}.{stg_tbl}", desc)

        gold_fqn = f"hive_metastore.{schema}.{gold_tbl}"
        df = _try_load(gold_fqn)
        if df is not None:
            seg_info["gold_rows"] = df.count()
            if "CaseNo" in df.columns:
                cached = df.select("CaseNo").distinct()
                cached.cache()
                _df_cache[f"gold:{label}"] = cached
                seg_info["gold"] = cached.count()
            else:
                seg_info["gold"] = seg_info["gold_rows"]
            dup_note = ""
            if seg_info["gold_rows"] != seg_info["gold"]:
                dup_note = f" (rows={seg_info['gold_rows']} — {seg_info['gold_rows'] - seg_info['gold']} duplicate CaseNo rows)"
            _add("test_gather_counts_A", f"{label}_gold", "PASS",
                 f"{seg_info['gold']} distinct CaseNos in {schema}.{gold_tbl}{dup_note}", desc)
        else:
            _add("test_gather_counts_A", f"{label}_gold", "NO_DATA",
                 f"could not load {schema}.{gold_tbl}", desc)

        A["archive_by_segment"][label] = seg_info

    # --- TD stg excluding 519 (for the spec-correct equation)
    A["TD_stg_excl_519"] = None
    td_stg = A["archive_by_segment"].get("TD", {}).get("stg")
    if isinstance(td_stg, int) and isinstance(A.get("dropped_519"), int):
        # TD stg currently includes 519s (dev-code deviation); subtract them for the spec-correct view.
        # This is informational; the strict equation test uses raw td_stg.
        A["TD_stg_excl_519"] = td_stg - A["dropped_519"]
        _add("test_gather_counts_A", "TD_stg_excl_519", "PASS",
             f"TD stg ({td_stg}) minus dept-519 ({A['dropped_519']}) = {A['TD_stg_excl_519']}", desc)

_run_timed("test_gather_counts_A", test_gather_counts_A)

In [0]:
def test_total_reconciliation_A():
    desc = ("raw_appealcase(CaseType=1) = active + |union of stg CaseNos across {FPA,FTA,UTA,TD}| "
            "(leftover computed against DISTINCT UNION to neutralise known structural overlap, "
            "e.g. FPA inside TD via DeptId=520 ELSE-branch routing)")
    A = counts["A"]
    abs_map = A.get("archive_by_segment", {})
    needed = {
        "raw_CT1":   A.get("raw_CT1"),
        "active":    A.get("active_stg"),
        "FPA_stg":   abs_map.get("FPA", {}).get("stg"),
        "FTA_stg":   abs_map.get("FTA", {}).get("stg"),
        "UTA_stg":   abs_map.get("UTA", {}).get("stg"),
        "TD_stg":    abs_map.get("TD",  {}).get("stg"),
    }
    missing = [k for k, v in needed.items() if v is None]
    if missing:
        _add("test_total_reconciliation_A", "EQUATION", "NO_DATA",
             f"missing inputs: {missing}", desc)
        return

    lhs = needed["raw_CT1"]
    rhs_arith = needed["active"] + needed["FPA_stg"] + needed["FTA_stg"] + needed["UTA_stg"] + needed["TD_stg"]

    # Build the DISTINCT UNION of archive stg CaseNos across {FPA, FTA, UTA, TD}.
    # The arithmetic sum double-counts known structural overlap (FPA inside TD via the ELSE-branch
    # in TD's CASE WHEN — see analysis note). The leftover should be computed against the union
    # so overlaps are neutralised; arithmetic sum is kept purely for the message.
    union_stg_distinct = None
    union_size = None
    seg_map = {label: (schema, stg) for label, schema, stg, _g in ARCHIVE_SEGMENTS}
    for label in EQUATION_SEGMENTS:
        if label not in seg_map: continue
        schema, stg_tbl = seg_map[label]
        if stg_tbl is None: continue
        fqn = f"hive_metastore.{schema}.{stg_tbl}"
        df = _try_load(fqn)
        if df is None or "CaseNo" not in df.columns:
            continue
        d = df.select("CaseNo").distinct()
        d.cache()
        _df_cache[f"stg:{label}"] = d
        union_stg_distinct = d if union_stg_distinct is None else union_stg_distinct.unionByName(d).distinct()

    if union_stg_distinct is not None:
        union_size = union_stg_distinct.distinct().count()
        _df_cache["A:union_stg_distinct"] = union_stg_distinct
        rhs_union = needed["active"] + union_size
        leftover = lhs - rhs_union
        msg = (f"raw={lhs} = active={needed['active']} + |union(FPA,FTA,UTA,TD) stg CaseNos|={union_size} "
               f"+ leftover={leftover}  (arithmetic-sum view: FPA={needed['FPA_stg']} + FTA={needed['FTA_stg']} "
               f"+ UTA={needed['UTA_stg']} + TD={needed['TD_stg']} = {needed['FPA_stg'] + needed['FTA_stg'] + needed['UTA_stg'] + needed['TD_stg']}; "
               f"arith - union = {(needed['FPA_stg'] + needed['FTA_stg'] + needed['UTA_stg'] + needed['TD_stg']) - union_size} structural overlap removed)")
    else:
        # Fallback: couldn't build union (stg tables missing CaseNo or unloadable) — use arithmetic.
        leftover = lhs - rhs_arith
        msg = (f"raw={lhs} = active={needed['active']} + FPA={needed['FPA_stg']} + FTA={needed['FTA_stg']} "
               f"+ UTA={needed['UTA_stg']} + TD={needed['TD_stg']} + leftover={leftover} "
               f"(union DF unavailable — arithmetic sum used)")

    # Tolerance for "leftover" — small positive residual is expected (skeleton/UT-active exclusions)
    # but a large gap is a real regression. Threshold: 1% of raw.
    leftover_pct = abs(leftover) / lhs * 100 if lhs else 0
    # 5% threshold — the post-union residual is dominated by structurally-excluded categories
    # (Skeleton Cases per CasePrefix in LP/LR/LD/LH/LE/IA + UT Active/Remitted) which every
    # archive segmentation explicitly drops. A genuine catastrophic regression would be well over 5%.
    PCT_THRESHOLD = 5.0

    if leftover == 0:
        _add("test_total_reconciliation_A", "EQUATION", "PASS", msg, desc)
    elif leftover < 0:
        _add("test_total_reconciliation_A", "EQUATION", "FAIL",
             f"NEGATIVE leftover means a case appears in active AND the archive union (cross-zone leakage). {msg}", desc)
    elif leftover_pct > PCT_THRESHOLD:
        _add("test_total_reconciliation_A", "EQUATION", "FAIL",
             f"leftover={leftover} ({leftover_pct:.2f}% of raw) exceeds {PCT_THRESHOLD}% threshold — pipeline may be dropping cases. {msg}", desc)
    else:
        _add("test_total_reconciliation_A", "EQUATION", "PASS",
             f"leftover={leftover} ({leftover_pct:.2f}% of raw — within {PCT_THRESHOLD}% threshold, likely skeleton/UT-active exclusions). {msg}", desc)

    A["leftover"] = leftover
    A["union_stg_size"] = union_size

_run_timed("test_total_reconciliation_A", test_total_reconciliation_A)

In [0]:
def test_within_active_uniqueness_A():
    desc = "A: no CaseNo should appear with more than one TargetState in stg_segmentation_states"
    seg_df = _try_load(ACTIVE_SEG_TBL)
    if seg_df is None:
        _add("test_within_active_uniqueness_A", "active", "NO_DATA", "active table not loadable", desc)
        return
    dups = (seg_df.filter(col("TargetState").isNotNull())
                  .groupBy("CaseNo")
                  .agg(F.countDistinct("TargetState").alias("n_states"),
                       F.collect_set("TargetState").alias("states"))
                  .filter(col("n_states") > 1))
    n = dups.count()
    counts["X"]["within_active_dup_states"] = n
    if n == 0:
        _add("test_within_active_uniqueness_A", "active", "PASS",
             "every CaseNo has exactly one TargetState", desc)
    else:
        examples = [(r.CaseNo, r.states) for r in dups.limit(5).collect()]
        _add("test_within_active_uniqueness_A", "active", "FAIL",
             f"{n} CaseNos in 2+ TargetStates; examples: {examples}", desc)

_run_timed("test_within_active_uniqueness_A", test_within_active_uniqueness_A)

In [0]:
def test_dept_519_isolation_A():
    desc = "A: no DeptId=519 case should appear in active or any archive stg/gold (spec assertion)"
    # Reuse the cached distinct-CaseNo DF from gather (avoids re-running the raw×filelocation join)
    dept519 = _df_cache.get("A:dept519_caseno")
    if dept519 is None:
        _add("test_dept_519_isolation_A", "raw_dept519", "NO_DATA",
             "dept-519 CaseNo set not cached — gather didn't run cleanly", desc)
        return
    n519 = counts["A"].get("dropped_519")
    _add("test_dept_519_isolation_A", "raw_dept519_count", "PASS",
         f"{n519} dept-519 cases identified in raw (cached from gather)", desc)

    # Active
    seg_df = _try_load(ACTIVE_SEG_TBL)
    if seg_df is not None:
        leaked = (seg_df.filter(col("TargetState").isNotNull())
                        .select("CaseNo").distinct()
                        .join(dept519, on="CaseNo", how="inner").count())
        _add("test_dept_519_isolation_A", "leaked_into_active",
             "PASS" if leaked == 0 else "FAIL",
             f"{leaked} dept-519 cases found in active (should be 0)", desc)

    # Each archive segment's stg + gold tables
    for label, schema, stg_tbl, gold_tbl in ARCHIVE_SEGMENTS:
        for kind, tbl in (("stg", stg_tbl), ("gold", gold_tbl)):
            if tbl is None: continue
            df = _try_load(f"hive_metastore.{schema}.{tbl}")
            if df is None: continue
            if "CaseNo" not in df.columns:
                _add("test_dept_519_isolation_A", f"{label}_{kind}", "NO_DATA",
                     "no CaseNo column to compare", desc)
                continue
            leaked = df.select("CaseNo").distinct().join(dept519, on="CaseNo", how="inner").count()
            status = "PASS" if leaked == 0 else "FAIL"
            note = ""
            if label == "TD" and leaked > 0:
                note = " — known dev-code deviation: ARIADM_ARM_TD.py routes 519s into TD archive"
            _add("test_dept_519_isolation_A", f"{label}_{kind}",
                 status,
                 f"{leaked} dept-519 cases found in {label} {kind} (should be 0){note}", desc)

_run_timed("test_dept_519_isolation_A", test_dept_519_isolation_A)

In [0]:
def test_dept_520_routing_A():
    desc = "A: all DeptId=520 cases should be in FPA archive and absent from active and other archive segments"
    dept520 = _df_cache.get("A:dept520_caseno")
    if dept520 is None:
        _add("test_dept_520_routing_A", "raw_dept520", "NO_DATA",
             "dept-520 CaseNo set not cached — gather didn't run cleanly", desc)
        return
    n520 = counts["A"].get("dept_520")
    _add("test_dept_520_routing_A", "raw_dept520_count", "PASS",
         f"{n520} dept-520 cases identified in raw (cached from gather)", desc)

    # Should NOT be in active
    seg_df = _try_load(ACTIVE_SEG_TBL)
    if seg_df is not None:
        leaked = (seg_df.filter(col("TargetState").isNotNull())
                        .select("CaseNo").distinct()
                        .join(dept520, on="CaseNo", how="inner").count())
        _add("test_dept_520_routing_A", "active_should_be_zero",
             "PASS" if leaked == 0 else "FAIL",
             f"{leaked} dept-520 cases in active (should be 0)", desc)

    # SHOULD all be in FPA; should NOT be in FTA/UTA/TD/BAIL/SBAIL
    for label, schema, stg_tbl, gold_tbl in ARCHIVE_SEGMENTS:
        df = _try_load(f"hive_metastore.{schema}.{gold_tbl}")
        if df is None or "CaseNo" not in df.columns: continue
        present = df.select("CaseNo").distinct().join(dept520, on="CaseNo", how="inner").count()
        if label == "FPA":
            ok = present == n520
            _add("test_dept_520_routing_A", "FPA_should_contain_all",
                 "PASS" if ok else "FAIL",
                 f"FPA gold contains {present} of {n520} dept-520 cases", desc)
        else:
            _add("test_dept_520_routing_A", f"{label}_should_be_zero",
                 "PASS" if present == 0 else "FAIL",
                 f"{present} dept-520 cases in {label} (should be 0)", desc)

_run_timed("test_dept_520_routing_A", test_dept_520_routing_A)

In [0]:
def test_active_per_state_gold_parity_A():
    """Per TargetState: load segmentation count, valid (gold) count, AND invalid (DQ-quarantine) count.
    Records this triplet in counts['A']['active_state_counts'][state] for downstream use by
    test_active_per_state_validation_balance_A and the flow summary printout.
    """
    desc = "A: per-TargetState seg / valid (gold) / invalid (DQ quarantine) counts"
    A = counts["A"]
    A["active_state_counts"] = {}
    seg_df = _try_load(ACTIVE_SEG_TBL)
    if seg_df is None:
        _add("test_active_per_state_gold_parity_A", "ALL", "NO_DATA",
             "active segmentation not loadable", desc)
        return

    # Per-state segmentation counts (one collect)
    seg_counts = {r.TargetState: r["count"] for r in
                  seg_df.filter(col("TargetState").isNotNull())
                        .groupBy("TargetState").count().collect()}

    def _distinct_caseno_count(df):
        if df is None: return None
        rows = df.count()
        if "CaseNo" in df.columns:
            return df.select("CaseNo").distinct().count(), rows
        return rows, rows

    for state, gold_fq in ACTIVE_STATE_TO_GOLD.items():
        seg_n = seg_counts.get(state)
        if seg_n is None:
            A["active_state_counts"][state] = {"seg": 0, "gold": None, "invalid": None}
            _add("test_active_per_state_gold_parity_A", state, "NO_DATA",
                 "TargetState not present in segmentation", desc)
            continue

        # Valid (gold) side
        gold_df = _try_load(gold_fq)
        if gold_df is None:
            A["active_state_counts"][state] = {"seg": seg_n, "gold": None, "invalid": None}
            _add("test_active_per_state_gold_parity_A", state, "NO_DATA",
                 f"gold table not loadable: {gold_fq}", desc)
            continue
        gold_n, gold_rows = _distinct_caseno_count(gold_df)

        # Invalid (DQ quarantine) side — load separately; may be empty/missing if no DQ failures
        invalid_fq = ACTIVE_STATE_TO_INVALID.get(state)
        invalid_n = None
        invalid_rows = None
        if invalid_fq:
            invalid_df = _try_load(invalid_fq)
            if invalid_df is not None:
                invalid_n, invalid_rows = _distinct_caseno_count(invalid_df)

        A["active_state_counts"][state] = {
            "seg": seg_n, "gold": gold_n, "gold_rows": gold_rows,
            "invalid": invalid_n, "invalid_rows": invalid_rows,
        }

    # Unmapped TargetStates
    for state, n in seg_counts.items():
        if state not in ACTIVE_STATE_TO_GOLD:
            A["active_state_counts"][state] = {"seg": n, "gold": None, "invalid": None, "unmapped": True}

    # ONE summary row instead of 16 per-state strict-parity rows. The real per-state assertion
    # lives in test_active_per_state_validation_balance_A which checks seg == valid + invalid.
    # (Strict seg==valid would always FAIL when DQ rejects anything; the balance check is the
    # meaningful one.)
    n_loaded = sum(1 for v in A["active_state_counts"].values() if v.get("gold") is not None)
    n_total  = len(ACTIVE_STATE_TO_GOLD)
    _add("test_active_per_state_gold_parity_A", "SUMMARY",
         "PASS" if n_loaded == n_total else "FAIL",
         f"{n_loaded}/{n_total} active gold tables loaded — per-state seg/valid/invalid stored for balance test", desc)

_run_timed("test_active_per_state_gold_parity_A", test_active_per_state_gold_parity_A)


def test_active_per_state_validation_balance_A():
    """For each TargetState, segmentation count must equal valid + invalid (DQ-rejected).
    This is the right end-to-end check: gold loses cases to DQ, but every segmentation case
    must land in EITHER the valid OR the invalid bucket. If neither, the pipeline lost the case."""
    desc = "A: per-state seg == valid + invalid (DQ-rejected). Catches silent case loss in the DLT gold step."
    A = counts["A"]
    active_state_counts = A.get("active_state_counts") or {}
    if not active_state_counts:
        _add("test_active_per_state_validation_balance_A", "ALL", "NO_DATA",
             "no per-state counts available (gather did not run)", desc)
        return

    A["total_invalid_quarantined"] = 0
    for state, vals in active_state_counts.items():
        if vals.get("unmapped"):
            continue  # no gold/invalid mapping known
        seg = vals.get("seg")
        valid = vals.get("gold")
        invalid = vals.get("invalid")
        if seg is None or valid is None:
            _add("test_active_per_state_validation_balance_A", state, "NO_DATA",
                 f"seg={seg} valid={valid} (one or both missing)", desc)
            continue
        # If the invalid table didn't load (NO_DATA in gather), treat as 0 for balance but flag
        inv_for_eq = invalid if isinstance(invalid, int) else 0
        if isinstance(invalid, int):
            A["total_invalid_quarantined"] += invalid

        diff = seg - (valid + inv_for_eq)
        if diff == 0:
            note = "" if isinstance(invalid, int) else " (invalid table didn't load — treated as 0)"
            _add("test_active_per_state_validation_balance_A", state, "PASS",
                 f"seg={seg} = valid={valid} + invalid={inv_for_eq}{note}", desc)
        else:
            _add("test_active_per_state_validation_balance_A", state, "FAIL",
                 f"seg={seg} != valid({valid}) + invalid({inv_for_eq}) — diff={diff} cases unaccounted for "
                 f"(silently dropped between segmentation and the DLT gold step)", desc)

_run_timed("test_active_per_state_validation_balance_A", test_active_per_state_validation_balance_A)

In [0]:
def test_archive_stg_pairwise_overlap_A():
    """Diagnostic: for each unordered pair in EQUATION_SEGMENTS ({FPA,FTA,UTA,TD}), intersect
    distinct CaseNos from each archive's stg table and report the overlap size.

    Pass/fail policy:
      - overlap == 0                                  -> PASS
      - overlap > 0 AND matches a known structural    -> PASS (with note)
      - overlap > 0 AND no known structural           -> FAIL
    """
    desc = ("A: pairwise distinct-CaseNo overlap across {FPA,FTA,UTA,TD} stg tables. "
            "Known structural overlap: FPA (DeptId=520) inside TD via the ELSE-branch of TD's CASE WHEN. "
            "Other pairs are by construction disjoint (stg_appealcasestatus_filtered drops DeptId=519/520; "
            "stg_appealcasestatus_filtered emits one CaseStatusCategory per CaseNo) — overlap there = code regression. "
            "FT/UT-skeleton vs the stg_appealcasestatus_filtered-mediated FT/UT segments uses an independent predicate "
            "and is a lower-likelihood diagnostic.")

    # Known structural overlaps — keys are frozensets of (label1, label2). Value is the human-readable rationale.
    KNOWN_STRUCTURAL = {
        frozenset({"FPA", "TD"}):
            "FPA (DeptId=520) is NOT excluded from TD stg — TD's CASE WHEN ELSE-branch routes "
            "any DeptId=520 CaseNo that doesn't satisfy Active-CCD or Retain-CCD into 'Tribunal Decision'. "
            "Expected double-counting; size should be <= FPA stg size.",
    }

    # Load distinct-CaseNo DFs from cache (populated by test_total_reconciliation_A) or freshly.
    seg_map = {label: (schema, stg) for label, schema, stg, _g in ARCHIVE_SEGMENTS}
    distincts = {}
    for label in EQUATION_SEGMENTS:
        if label not in seg_map: continue
        schema, stg_tbl = seg_map[label]
        if stg_tbl is None: continue
        cached = _df_cache.get(f"stg:{label}")
        if cached is None:
            fqn = f"hive_metastore.{schema}.{stg_tbl}"
            df = _try_load(fqn)
            if df is None or "CaseNo" not in df.columns:
                _add("test_archive_stg_pairwise_overlap_A", f"{label}_load", "NO_DATA",
                     f"could not load distinct CaseNos for {label} stg ({fqn})", desc)
                continue
            cached = df.select("CaseNo").distinct()
            cached.cache()
            _df_cache[f"stg:{label}"] = cached
        distincts[label] = cached

    if len(distincts) < 2:
        _add("test_archive_stg_pairwise_overlap_A", "pairwise", "NO_DATA",
             "fewer than 2 stg tables loadable — cannot run pairwise overlap", desc)
        return

    labels = sorted(distincts.keys())
    counts["A"].setdefault("pairwise_stg_overlap", {})
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            a, b = labels[i], labels[j]
            field = f"{a}_vs_{b}"
            df_a = distincts[a]
            df_b = distincts[b]
            overlap_df = df_a.join(df_b, on="CaseNo", how="inner")
            overlap = overlap_df.count()
            counts["A"]["pairwise_stg_overlap"][field] = overlap

            structural_note = KNOWN_STRUCTURAL.get(frozenset({a, b}))
            if overlap == 0:
                _add("test_archive_stg_pairwise_overlap_A", field, "PASS",
                     f"{a} stg intersect {b} stg = 0 distinct CaseNos", desc)
            elif structural_note is not None:
                examples = [r.CaseNo for r in overlap_df.limit(3).collect()]
                _add("test_archive_stg_pairwise_overlap_A", field, "PASS",
                     f"{a} stg intersect {b} stg = {overlap} distinct CaseNos — KNOWN STRUCTURAL OVERLAP: "
                     f"{structural_note} Examples: {examples}", desc)
            else:
                examples = [r.CaseNo for r in overlap_df.limit(5).collect()]
                _add("test_archive_stg_pairwise_overlap_A", field, "FAIL",
                     f"{a} stg intersect {b} stg = {overlap} distinct CaseNos — NOT a known structural overlap "
                     f"(stg_appealcasestatus_filtered should keep these disjoint). Examples: {examples}", desc)

_run_timed("test_archive_stg_pairwise_overlap_A", test_archive_stg_pairwise_overlap_A)

In [0]:
def test_archive_stg_vs_gold_parity_A():
    desc = "A: for each archive segment with a known stg, count(stg) == count(gold)"
    abs_map = counts["A"].get("archive_by_segment", {})
    for label, schema, stg_tbl, gold_tbl in ARCHIVE_SEGMENTS:
        if stg_tbl is None:
            continue  # BAIL/SBAIL handled in their own universes
        info = abs_map.get(label, {})
        s = info.get("stg")
        g = info.get("gold")
        if s is None or g is None:
            _add("test_archive_stg_vs_gold_parity_A", label, "NO_DATA",
                 f"stg={s} gold={g} (one or both not loaded)", desc)
            continue
        diff = s - g
        if diff == 0:
            _add("test_archive_stg_vs_gold_parity_A", label, "PASS",
                 f"stg={s} gold={g} match", desc)
        else:
            _add("test_archive_stg_vs_gold_parity_A", label, "FAIL",
                 f"stg={s} gold={g} diff={diff} — downstream pipeline lost/added rows", desc)

_run_timed("test_archive_stg_vs_gold_parity_A", test_archive_stg_vs_gold_parity_A)

## Bails data — BAIL (aria_bails.raw_appeal_cases, CaseType=2)

BAIL archives cases from the `aria_bails` schema. The raw source is `raw_appeal_cases` filtered by `CaseType=2 AND DeptId != 519 AND Note NOT destroyed`.

```
raw_appeal_cases WHERE CaseType=2
  -> (dev-code filter: DeptId != 519, Note NOT destroyed-variants)
  -> silver_bail_combined_segmentation_nb_lhnb
  ~= create_bails_json_content   (gold)
```

All results in this section are tagged `test_from_state = bail_reconciliation`.

In [0]:
_set_scope("bail_reconciliation")

def test_gather_counts_B():
    desc = "B: BAIL raw / filter / segmentation / gold counts"
    B = counts["B"]

    raw_df = _try_load(BAIL_RAW_TBL)
    fl_df  = _try_load(BAIL_FL_TBL)

    if raw_df is None:
        _add("test_gather_counts_B", "raw", "NO_DATA", f"could not load {BAIL_RAW_TBL}", desc)
        B["raw_CT2"] = None
    else:
        B["raw_CT2"] = raw_df.filter(col("CaseType") == 2).select("CaseNo").distinct().count()
        _add("test_gather_counts_B", "raw", "PASS",
             f"{B['raw_CT2']} distinct CaseNos with CaseType=2 in raw", desc)

    # Dev-code segmentation filter (mirrors Dynamic-ARIA-Bails v2.py)
    if raw_df is not None and fl_df is not None:
        joined = raw_df.alias("ac").join(fl_df.alias("fl"), col("ac.CaseNo") == col("fl.CaseNo"), "left_outer")
        passing = joined.filter(
            (col("ac.CaseType") == 2)
            & ((col("fl.DeptId") != 519) | col("fl.DeptId").isNull())
            & (
                (
                    (~col("fl.Note").like("%destroyed%"))
                    & (~col("fl.Note").like("%detroyed%"))
                    & (~col("fl.Note").like("%destoyed%"))
                    & (~col("fl.Note").like("%distroyed%"))
                )
                | col("fl.Note").isNull()
            )
        )
        B["after_filter"] = passing.select(col("ac.CaseNo")).distinct().count()
        _add("test_gather_counts_B", "after_filter", "PASS",
             f"{B['after_filter']} CaseNos pass dev-code segmentation filter", desc)
    else:
        B["after_filter"] = None

    seg_df = _try_load(BAIL_SEG_TBL)
    if seg_df is None:
        _add("test_gather_counts_B", "segmentation_table", "NO_DATA",
             f"{BAIL_SEG_TBL} not loadable", desc)
        B["seg"] = None
    else:
        B["seg"] = seg_df.select("CaseNo").distinct().count()
        _add("test_gather_counts_B", "segmentation_table", "PASS",
             f"{B['seg']} distinct CaseNos in silver_bail_combined_segmentation_nb_lhnb", desc)

    # Gold — also cache distinct CaseNos for cross-cutting reuse
    gold_distinct = _load_distinct_caseno(BAIL_GOLD_TBL, cache_key="gold:BAIL")
    if gold_distinct is None:
        gold_df = _try_load(BAIL_GOLD_TBL)
        if gold_df is None:
            _add("test_gather_counts_B", "gold", "NO_DATA",
                 f"{BAIL_GOLD_TBL} not loadable", desc)
            B["gold"] = None
        else:
            B["gold"] = gold_df.count()
            _add("test_gather_counts_B", "gold", "PASS",
                 f"{B['gold']} rows in create_bails_json_content (no CaseNo column for distinct cache)", desc)
    else:
        gold_df = _try_load(BAIL_GOLD_TBL)
        B["gold"] = gold_df.count() if gold_df is not None else None
        _add("test_gather_counts_B", "gold", "PASS",
             f"{B['gold']} rows in create_bails_json_content", desc)

_run_timed("test_gather_counts_B", test_gather_counts_B)

In [0]:
def test_reconciliation_B():
    desc = "B: BAIL segmentation count must equal gold count"
    s, g = counts["B"].get("seg"), counts["B"].get("gold")
    if s is None or g is None:
        _add("test_reconciliation_B", "EQUATION", "NO_DATA",
             "segmentation or gold count missing", desc)
        return
    diff = s - g
    _add("test_reconciliation_B", "EQUATION",
         "PASS" if diff == 0 else "FAIL",
         f"segmentation={s} gold={g} diff={diff}", desc)

def test_filter_vs_segmentation_B():
    desc = ("B: informational diff between dev-code filter (CT=2, DeptId!=519, Note NOT destroyed) "
            "and the segmentation table — segmentation applies further routing logic beyond the basic filter")
    f, s = counts["B"].get("after_filter"), counts["B"].get("seg")
    if f is None or s is None:
        _add("test_filter_vs_segmentation_B", "filter_vs_seg", "NO_DATA", "missing counts", desc)
        return
    diff = f - s
    # Informational only: the BAIL pipeline legitimately drops ~95% of filter-passing cases via
    # Legal Hold / Destroy / Archive routing downstream of the basic filter, so a large gap
    # (often >20x) is EXPECTED. We report the numbers but never fail on this diff — segmentation
    # applies further routing logic beyond the basic filter.
    _add("test_filter_vs_segmentation_B", "filter_vs_seg", "PASS",
         f"filter={f} seg={s} diff={diff} (informational — segmentation applies further routing logic beyond the basic filter)", desc)

_run_timed("test_reconciliation_B", test_reconciliation_B)
_run_timed("test_filter_vs_segmentation_B", test_filter_vs_segmentation_B)

In [0]:
def test_gold_uniqueness_B():
    desc = "B: every BAIL gold row has a unique CaseNo"
    gold_df = _try_load(BAIL_GOLD_TBL)
    if gold_df is None:
        _add("test_gold_uniqueness_B", "gold", "NO_DATA", "gold not loadable", desc)
        return
    total = gold_df.count()
    if "CaseNo" not in gold_df.columns:
        _add("test_gold_uniqueness_B", "gold", "NO_DATA", "no CaseNo column in gold", desc)
        return
    distinct = gold_df.select("CaseNo").distinct().count()
    if total == distinct:
        _add("test_gold_uniqueness_B", "gold", "PASS", f"{total} rows, all CaseNos unique", desc)
    else:
        dups = (gold_df.groupBy("CaseNo").count().filter(col("count") > 1).limit(5).collect())
        ex = [(r.CaseNo, r["count"]) for r in dups]
        _add("test_gold_uniqueness_B", "gold", "FAIL",
             f"total={total} distinct={distinct} duplicate examples: {ex}", desc)

_run_timed("test_gold_uniqueness_B", test_gold_uniqueness_B)

## Scottish Bails data — SBAIL (Scottish__Bailsfile.csv -> silver_scottish_sbails_funds)

Unlike BAIL, the source is **not** a database raw table — it's an external CSV file (`Scottish__Bailsfile.csv`) listing the gating CaseNos. The SBAIL pipeline loads that CSV into `silver_scottish_sbails_funds`, then routes each listed CaseNo into the gold table.

```
silver_scottish_sbails_funds   (loaded from Scottish__Bailsfile.csv)
  ~= create_sbail_json_content   (gold)
```

Simplest reconciliation in the suite — fixed list in, transformed list out, count must match.

All results in this section are tagged `test_from_state = sbail_reconciliation`.

In [0]:
_set_scope("sbail_reconciliation")

def test_gather_counts_C():
    desc = "C: SBAIL CSV-source / segmentation / gold counts"
    C = counts["C"]

    seg_df = _try_load(SBAIL_SEG_TBL)
    if seg_df is None:
        _add("test_gather_counts_C", "csv_segmentation", "NO_DATA",
             f"{SBAIL_SEG_TBL} not loadable", desc)
        C["seg"] = None
    else:
        C["seg"] = seg_df.select("CaseNo").distinct().count()
        # CSV non-empty assertion — a failed CSV load presents as 0 rows here, which would let
        # reconciliation pass vacuously (0 == 0). Floor it explicitly.
        SBAIL_MIN_FLOOR = 100   # conservative — adjust to known size if Scottish bails ever shrinks below this
        if C["seg"] == 0:
            _add("test_gather_counts_C", "csv_non_empty", "FAIL",
                 f"silver_scottish_sbails_funds is EMPTY — Scottish__Bailsfile.csv likely failed to load. "
                 f"All downstream SBAIL checks will pass vacuously and hide the failure.", desc)
        elif C["seg"] < SBAIL_MIN_FLOOR:
            _add("test_gather_counts_C", "csv_non_empty", "FAIL",
                 f"silver_scottish_sbails_funds has only {C['seg']} rows — below floor of {SBAIL_MIN_FLOOR}. "
                 f"CSV may have loaded partially.", desc)
        else:
            _add("test_gather_counts_C", "csv_non_empty", "PASS",
                 f"silver_scottish_sbails_funds has {C['seg']} rows (>= floor {SBAIL_MIN_FLOOR})", desc)
        _add("test_gather_counts_C", "csv_segmentation", "PASS",
             f"{C['seg']} distinct CaseNos in silver_scottish_sbails_funds (from Scottish__Bailsfile.csv)",
             desc)

    # Gold — cache distinct CaseNos for cross-cutting reuse
    gold_distinct = _load_distinct_caseno(SBAIL_GOLD_TBL, cache_key="gold:SBAIL")
    gold_df = _try_load(SBAIL_GOLD_TBL)
    if gold_df is None:
        _add("test_gather_counts_C", "gold", "NO_DATA", f"{SBAIL_GOLD_TBL} not loadable", desc)
        C["gold"] = None
    else:
        C["gold"] = gold_df.count()
        _add("test_gather_counts_C", "gold", "PASS",
             f"{C['gold']} rows in create_sbail_json_content", desc)

_run_timed("test_gather_counts_C", test_gather_counts_C)

In [0]:
def test_reconciliation_C():
    desc = "C: SBAIL CSV-driven segmentation count must equal gold count"
    s, g = counts["C"].get("seg"), counts["C"].get("gold")
    if s is None or g is None:
        _add("test_reconciliation_C", "EQUATION", "NO_DATA",
             "segmentation or gold count missing", desc)
        return
    diff = s - g
    _add("test_reconciliation_C", "EQUATION",
         "PASS" if diff == 0 else "FAIL",
         f"segmentation={s} gold={g} diff={diff}", desc)

_run_timed("test_reconciliation_C", test_reconciliation_C)

In [0]:
def test_gold_uniqueness_C():
    desc = "C: every SBAIL gold row has a unique CaseNo"
    gold_df = _try_load(SBAIL_GOLD_TBL)
    if gold_df is None:
        _add("test_gold_uniqueness_C", "gold", "NO_DATA", "gold not loadable", desc)
        return
    total = gold_df.count()
    if "CaseNo" not in gold_df.columns:
        _add("test_gold_uniqueness_C", "gold", "NO_DATA", "no CaseNo column in gold", desc)
        return
    distinct = gold_df.select("CaseNo").distinct().count()
    if total == distinct:
        _add("test_gold_uniqueness_C", "gold", "PASS", f"{total} rows, all CaseNos unique", desc)
    else:
        dups = (gold_df.groupBy("CaseNo").count().filter(col("count") > 1).limit(5).collect())
        ex = [(r.CaseNo, r["count"]) for r in dups]
        _add("test_gold_uniqueness_C", "gold", "FAIL",
             f"total={total} distinct={distinct} duplicate examples: {ex}", desc)

_run_timed("test_gold_uniqueness_C", test_gold_uniqueness_C)

## Cross-cutting checks (A + B + C)

A CaseNo should appear in exactly ONE zone. Checked against every CaseNo-keyed archive segment (FPA, FTA, UTA, BAIL, SBAIL, TD) and inside the active universe itself.

All results in this section are tagged `test_from_state = overall_cross_cutting`. Reuses the distinct-CaseNo DataFrames cached during the per-universe gather steps.

In [0]:
_set_scope("overall_cross_cutting")

def test_cross_zone_uniqueness_X():
    desc = "no CaseNo should appear in both active segmentation and any archive gold (excl. JOH)"

    active_cases = _df_cache.get("A:active_distinct")
    if active_cases is None:
        seg_df = _try_load(ACTIVE_SEG_TBL)
        if seg_df is None:
            _add("test_cross_zone_uniqueness_X", "active_vs_archive", "NO_DATA",
                 "active table not loadable", desc)
            return
        active_cases = seg_df.filter(col("TargetState").isNotNull()).select("CaseNo").distinct()
        active_cases.cache()
        _df_cache["A:active_distinct"] = active_cases

    active_count = active_cases.count()

    seg_map = {label: (schema, gold) for label, schema, _stg, gold in ARCHIVE_SEGMENTS}
    for label in ARCHIVE_CASENO_SEGMENTS:
        if label not in seg_map:
            continue
        schema, gold_tbl = seg_map[label]
        gold_fqn = f"hive_metastore.{schema}.{gold_tbl}"
        archive_cases = _df_cache.get(f"gold:{label}")
        if archive_cases is None:
            archive_cases = _load_distinct_caseno(gold_fqn, cache_key=f"gold:{label}")
        if archive_cases is None:
            _add("test_cross_zone_uniqueness_X", f"active_vs_{label}", "NO_DATA",
                 f"{label} archive not loadable (or no CaseNo column)", desc)
            continue
        overlap = active_cases.join(archive_cases, on="CaseNo", how="inner")
        n = overlap.count()
        counts["X"][f"active_vs_{label}_overlap"] = n
        if n == 0:
            _add("test_cross_zone_uniqueness_X", f"active_vs_{label}", "PASS",
                 f"no overlap (active={active_count}, {label}={archive_cases.count()})", desc)
        else:
            examples = [r.CaseNo for r in overlap.limit(5).collect()]
            _add("test_cross_zone_uniqueness_X", f"active_vs_{label}", "FAIL",
                 f"{n} CaseNos in both zones; examples: {examples}", desc)

_run_timed("test_cross_zone_uniqueness_X", test_cross_zone_uniqueness_X)

In [0]:
def test_within_archive_uniqueness_X():
    desc = "no CaseNo should appear in two of the CaseNo-keyed archive gold tables"

    seg_map = {label: (schema, gold) for label, schema, _stg, gold in ARCHIVE_SEGMENTS}
    case_dfs = {}
    for label in ARCHIVE_CASENO_SEGMENTS:
        if label not in seg_map: continue
        schema, gold_tbl = seg_map[label]
        gold_fqn = f"hive_metastore.{schema}.{gold_tbl}"
        cached = _df_cache.get(f"gold:{label}")
        if cached is None:
            cached = _load_distinct_caseno(gold_fqn, cache_key=f"gold:{label}")
        if cached is None: continue
        case_dfs[label] = cached.withColumn("_seg", F.lit(label))
    if len(case_dfs) < 2:
        _add("test_within_archive_uniqueness_X", "archive", "NO_DATA",
             "need at least 2 archive segments loadable", desc)
        return

    unioned = None
    for df in case_dfs.values():
        unioned = df if unioned is None else unioned.unionByName(df)
    dups = (unioned.groupBy("CaseNo")
                   .agg(F.countDistinct("_seg").alias("n_segs"),
                        F.collect_set("_seg").alias("segs"))
                   .filter(col("n_segs") > 1))
    n = dups.count()
    counts["X"]["within_archive_overlap"] = n
    if n == 0:
        _add("test_within_archive_uniqueness_X", "archive", "PASS",
             f"no CaseNo in 2+ of {list(case_dfs.keys())}", desc)
    else:
        examples = [(r.CaseNo, r.segs) for r in dups.limit(5).collect()]
        _add("test_within_archive_uniqueness_X", "archive", "FAIL",
             f"{n} CaseNos in multiple segments; examples: {examples}", desc)

_run_timed("test_within_archive_uniqueness_X", test_within_archive_uniqueness_X)

## Combined flow summary

ASCII tree showing the split from `raw_appealcase`, the BAIL CaseType=2 branch, the SBAIL CSV branch, and the cross-cutting overlap counts. Each visible row also lands as a `flow_summary` TestResult so CSV/XLSX/HTML capture the full breakdown.

In [0]:
_set_scope("overall_flow_summary")

def emit_flow_summary_combined():
    A = counts["A"]; B = counts["B"]; C = counts["C"]; X = counts["X"]
    abs_map  = A.get("archive_by_segment", {}) or {}

    # ---- Appeals data leg
    raw_A    = A.get("raw_CT1")
    actv     = A.get("active_stg")
    fpa_s    = abs_map.get("FPA", {}).get("stg")
    fta_s    = abs_map.get("FTA", {}).get("stg")
    uta_s    = abs_map.get("UTA", {}).get("stg")
    td_s     = abs_map.get("TD",  {}).get("stg")
    td_excl  = A.get("TD_stg_excl_519")
    d519     = A.get("dropped_519")
    d520     = A.get("dept_520")
    leftover = A.get("leftover")

    # ---- Bails data leg
    raw_B    = B.get("raw_CT2")
    flt_B    = B.get("after_filter")
    seg_B    = B.get("seg")
    gold_B   = B.get("gold")

    # ---- Scottish Bails data leg
    seg_C    = C.get("seg")
    gold_C   = C.get("gold")

    desc_A = "A: raw_appealcase(CaseType=1) split into routing buckets"
    _add("flow_summary", "A_01_raw_appealcase_CT1",  "PASS" if isinstance(raw_A, int)  else "NO_DATA",
         f"raw_appealcase WHERE CaseType=1: {_fmt_n(raw_A).strip()}", desc_A)
    _add("flow_summary", "A_02_active",              "PASS" if isinstance(actv, int)   else "NO_DATA",
         f"active stg_segmentation_states: {_fmt_n(actv).strip()}", desc_A)
    _add("flow_summary", "A_03_FPA_stg_520_bucket",  "PASS" if isinstance(fpa_s, int)  else "NO_DATA",
         f"FPA stg_filepreservedcases_filtered: {_fmt_n(fpa_s).strip()}", desc_A)
    _add("flow_summary", "A_04_FTA_stg",             "PASS" if isinstance(fta_s, int)  else "NO_DATA",
         f"FTA stg_appeals_filtered: {_fmt_n(fta_s).strip()}", desc_A)
    _add("flow_summary", "A_05_UTA_stg",             "PASS" if isinstance(uta_s, int)  else "NO_DATA",
         f"UTA stg_appeals_filtered: {_fmt_n(uta_s).strip()}", desc_A)
    _add("flow_summary", "A_06_TD_stg_total",        "PASS" if isinstance(td_s, int)   else "NO_DATA",
         f"TD stg_td_filtered (includes 519s currently): {_fmt_n(td_s).strip()}", desc_A)
    _add("flow_summary", "A_07_TD_stg_excluding_519","PASS" if isinstance(td_excl, int) else "NO_DATA",
         f"TD stg minus dept-519 (spec-correct content): {_fmt_n(td_excl).strip()}", desc_A)
    _add("flow_summary", "A_08_dropped_519",         "PASS" if isinstance(d519, int)   else "NO_DATA",
         f"DeptId=519 (should be left behind): {_fmt_n(d519).strip()}", desc_A)
    _add("flow_summary", "A_09_dept_520_informational","PASS" if isinstance(d520, int) else "NO_DATA",
         f"DeptId=520 (informational; same set as FPA stg): {_fmt_n(d520).strip()}", desc_A)
    _add("flow_summary", "A_10_leftover", "PASS" if isinstance(leftover, int) else "NO_DATA",
         f"raw - (active+FPA+FTA+UTA+TD): {_fmt_n(leftover).strip()}", desc_A)
    total_inv = (counts.get("A") or {}).get("total_invalid_quarantined")
    _add("flow_summary", "A_11_dq_quarantined_total",
         "PASS" if isinstance(total_inv, int) else "NO_DATA",
         f"DQ-quarantined across active gold tables (stg_invalid_*): {_fmt_n(total_inv).strip()}",
         "Total cases dropped by DLT expectations at the active gold step (seg = valid + invalid)")

    # Per-segment gold row counts
    for label, schema, _stg, gold in ARCHIVE_SEGMENTS:
        n = abs_map.get(label, {}).get("gold")
        if n is None:
            _add("flow_summary", f"A_archive_gold:{label}", "NO_DATA",
                 f"{schema}.{gold} not loadable", "Archive segment gold count")
        else:
            _add("flow_summary", f"A_archive_gold:{label}", "PASS",
                 f"{label} gold rows: {n}", "Archive segment gold count")

    desc_B = "B: raw_appeal_cases(CaseType=2) -> dev-code filter -> segmentation -> gold"
    _add("flow_summary", "B_01_raw_CT2",         "PASS" if isinstance(raw_B, int)  else "NO_DATA",
         f"raw_appeal_cases CaseType=2: {_fmt_n(raw_B).strip()}", desc_B)
    _add("flow_summary", "B_02_after_filter",    "PASS" if isinstance(flt_B, int)  else "NO_DATA",
         f"after segmentation filter: {_fmt_n(flt_B).strip()}", desc_B)
    _add("flow_summary", "B_03_segmentation",    "PASS" if isinstance(seg_B, int)  else "NO_DATA",
         f"silver_bail_combined_segmentation_nb_lhnb: {_fmt_n(seg_B).strip()}", desc_B)
    _add("flow_summary", "B_04_gold",            "PASS" if isinstance(gold_B, int) else "NO_DATA",
         f"create_bails_json_content: {_fmt_n(gold_B).strip()}", desc_B)
    if isinstance(seg_B, int) and isinstance(gold_B, int):
        _add("flow_summary", "B_05_reconciliation",
             "PASS" if (seg_B - gold_B) == 0 else "FAIL",
             f"segmentation - gold = {seg_B - gold_B}", desc_B)

    desc_C = "C: Scottish__Bailsfile.csv -> silver_scottish_sbails_funds -> gold"
    _add("flow_summary", "C_01_csv_segmentation", "PASS" if isinstance(seg_C, int)  else "NO_DATA",
         f"silver_scottish_sbails_funds (CSV-loaded): {_fmt_n(seg_C).strip()}", desc_C)
    _add("flow_summary", "C_02_gold",             "PASS" if isinstance(gold_C, int) else "NO_DATA",
         f"create_sbail_json_content: {_fmt_n(gold_C).strip()}", desc_C)
    if isinstance(seg_C, int) and isinstance(gold_C, int):
        _add("flow_summary", "C_03_reconciliation",
             "PASS" if (seg_C - gold_C) == 0 else "FAIL",
             f"segmentation - gold = {seg_C - gold_C}", desc_C)

    desc_X = "X: cross-cutting overlap counts (active intersect each archive, within-archive duplicates)"
    for k, v in X.items():
        _add("flow_summary", f"X_{k}",
             "PASS" if (isinstance(v, int) and v == 0) else ("NO_DATA" if v is None else "FAIL"),
             f"{k}: {v}", desc_X)

    # ---- ASCII tree print
    def line(label, n, dots=60):
        pad = max(2, dots - len(label))
        return f"  {label} {'.' * pad} {_fmt_n(n)}"

    print()
    print("=" * 92)
    print("  COMBINED FLOW SUMMARY")
    print("=" * 92)
    print()
    print("  Appeals data — Active appeals (raw_appealcase, CaseType=1)")
    print("  " + "-" * 88)
    print(line("raw_appealcase WHERE CaseType=1", raw_A))
    print( "  |")
    print(line("+- Active (stg_segmentation_states, TargetState NOT NULL)", actv, 55))
    print( "  |")
    print(line("+- FPA stg (file-preserved, DeptId=520 bucket)", fpa_s, 55))
    print(line("+- FTA stg (first-tier appeals master union)", fta_s, 55))
    print(line("+- UTA stg (upper-tribunal master union)", uta_s, 55))
    print(line("+- TD stg total (includes dept-519s currently)", td_s, 55))
    print(line("|    of which dept-519 (SPEC VIOLATION: should be 0 in TD)", d519, 60))
    print(line("|    remainder after excluding 519 (TD spec-correct)", td_excl, 60))
    print( "  |")
    print(line("+- dropped_519 (left-behind per spec)", d519, 55))
    total_inv = (counts.get("A") or {}).get("total_invalid_quarantined")
    if isinstance(total_inv, int):
        print(line("+- DQ-quarantined across active gold tables (stg_invalid_*)", total_inv, 60))
    print(line("+- leftover (raw - rhs)", leftover, 55))
    print()

    # ---- Per-TargetState breakdown of the active bucket
    active_state_counts = (counts.get("A") or {}).get("active_state_counts") or {}
    if active_state_counts:
        print(f"  Appeals data breakdown — Active by TargetState (valid + DQ-quarantined = segmentation)")
        print("  " + "-" * 96)
        active_total_seg = 0
        active_total_valid = 0
        active_total_invalid = 0
        for state in sorted(active_state_counts.keys()):
            vals = active_state_counts[state]
            seg = vals.get("seg") or 0
            valid = vals.get("gold")
            invalid = vals.get("invalid")
            unmapped = vals.get("unmapped", False)
            active_total_seg += seg
            if isinstance(valid, int):   active_total_valid   += valid
            if isinstance(invalid, int): active_total_invalid += invalid

            if unmapped:
                tag = "  [UNMAPPED]"
            elif valid is None:
                tag = "  [valid MISSING]"
            else:
                inv_eq = invalid if isinstance(invalid, int) else 0
                balance = seg - (valid + inv_eq)
                if balance == 0:
                    tag = "  [OK]" + ("" if isinstance(invalid, int) else " (invalid n/a)")
                else:
                    tag = f"  [MISMATCH unaccounted={balance}]"

            valid_str   = f"{valid:>10,}"   if isinstance(valid, int)   else "         ?"
            invalid_str = f"{invalid:>8,}"  if isinstance(invalid, int) else "       ?"
            print(f"    {state:<35} seg={seg:>10,}  valid={valid_str}  invalid={invalid_str}{tag}")

        print(f"    {'':<35} {'-'*44}")
        print(f"    {'TOTAL across all TargetStates':<35} "
              f"seg={active_total_seg:>10,}  valid={active_total_valid:>10,}  "
              f"invalid={active_total_invalid:>8,}"
              f"  (delta={active_total_seg - active_total_valid - active_total_invalid})")
        print()

    print("  Bails data — BAIL (aria_bails.raw_appeal_cases, CaseType=2)")
    print("  " + "-" * 88)
    print(line("raw_appeal_cases WHERE CaseType=2", raw_B))
    print( "  |   filter: DeptId != 519 AND Note NOT destroyed")
    print(line("after dev-code filter", flt_B, 55))
    print( "  |")
    print(line("silver_bail_combined_segmentation_nb_lhnb", seg_B, 55))
    print( "  |   pipeline transformation")
    print(line("create_bails_json_content (gold)", gold_B, 55))
    if isinstance(seg_B, int) and isinstance(gold_B, int):
        tick = "OK" if (seg_B - gold_B) == 0 else "MISMATCH"
        print(f"    -> reconciliation diff: {seg_B - gold_B}  [{tick}]")
    print()

    print("  Scottish Bails data — SBAIL (Scottish__Bailsfile.csv)")
    print("  " + "-" * 88)
    print( "  Scottish__Bailsfile.csv")
    print( "    | loaded into silver_scottish_sbails_funds")
    print(line("silver_scottish_sbails_funds (gating list)", seg_C, 55))
    print( "  |   pipeline transformation")
    print(line("create_sbail_json_content (gold)", gold_C, 55))
    if isinstance(seg_C, int) and isinstance(gold_C, int):
        tick = "OK" if (seg_C - gold_C) == 0 else "MISMATCH"
        print(f"    -> reconciliation diff: {seg_C - gold_C}  [{tick}]")
    print()

    print("  Cross-cutting (X) — overlap counts (all should be 0)")
    print("  " + "-" * 88)
    if not X:
        print("    (no cross-cutting counts captured)")
    else:
        for k, v in X.items():
            print(line(k, v, 55))
    print()
    print("=" * 92)

_run_timed("flow_summary", emit_flow_summary_combined)

## Save results & headline banner

Writes ONE row to `test_automation_runs2` covering all three universes plus the cross-cutting + flow checks, then all `TestResult` rows (each carrying its own `test_from_state` so downstream queries can split by universe). CSV/XLSX/HTML files write to the user's Results folder.

In [0]:
# Tag the overall summary as "overall_reconciliation" so the banner row is easy to spot.
_set_scope("overall_reconciliation")

status_counts = count_by_status(all_test_results)
n_fail   = status_counts.get("FAIL", 0)
n_err    = status_counts.get("ERROR", 0)
n_pass   = status_counts.get("PASS", 0)
n_nodata = status_counts.get("NO_DATA", 0)
n_total  = status_counts.get("TOTAL", 0)
elapsed  = sum(p["elapsed_seconds"] for p in perf_timings)

overall = "PASS" if (n_fail == 0 and n_err == 0) else "FAIL"
banner_msg = (f"Total={n_total} Pass={n_pass} Fail={n_fail} Error={n_err} "
              f"NoData={n_nodata} Elapsed={elapsed:.1f}s")

# Per-universe breakdown line (PASS/FAIL counts per test_from_state)
by_scope = {}
for r in all_test_results:
    s = by_scope.setdefault(r.test_from_state, {"PASS": 0, "FAIL": 0, "ERROR": 0, "NO_DATA": 0})
    s[r.status] = s.get(r.status, 0) + 1
print()
print("Per-universe breakdown:")
for scope, s in sorted(by_scope.items()):
    print(f"  {scope:35s}  PASS={s.get('PASS',0):4d}  FAIL={s.get('FAIL',0):4d}  "
          f"ERROR={s.get('ERROR',0):4d}  NO_DATA={s.get('NO_DATA',0):4d}")

# Single banner row — scope back to overall_reconciliation so the banner is tagged as overall
all_test_results.append(TestResult(
    test_name="00_OVERALL_SUMMARY", test_field="00_OVERALL", test_from_state="overall_reconciliation",
    status=overall, message=banner_msg,
    description="Overall verdict for the consolidated reconciliation run "
                "(covers active + BAIL + SBAIL + cross-cutting).",
))

# Persist — one run_id covers everything
run_id = create_run(spark, run_user, run_start_datetime, "Overall_Reconciliation_Tests",
                    run_tag, overall, "overall_reconciliation")
n_results = create_results(spark, run_id, all_test_results)
n_perf    = create_perf_results(spark, run_id, perf_timings, "overall_reconciliation")

# CSV-only output — single canonical artefact for the reconciliation report.
# (XLSX and HTML formatters intentionally not used here; the dashboards already query
# test_automation_results2 for the visual layer.)
folder, ts, _safe = build_report_folder(test_results_path, "overall_reconciliation")
try:
    write_csv(all_test_results, "overall_reconciliation", folder, ts)
except Exception as _e:
    print(f"  ! csv write failed: {type(_e).__name__}: {str(_e)[:160]}")

print("=" * 70)
print(f"  OVERALL RESULT: {overall}   (consolidated A + B + C + cross-cutting)")
print("=" * 70)
print(f"  {banner_msg}")
print(f"  run_id: {run_id}   results written: {n_results}   perf rows: {n_perf}")
print(f"  Folder: {folder}")
print("=" * 70)

# Inline failures (skip the banner row to avoid noise)
for r in all_test_results:
    if r.status in ("FAIL", "ERROR") and r.test_name != "00_OVERALL_SUMMARY":
        print(f"    [{r.test_from_state}] [{r.test_name}] {r.test_field} -- {r.message}")

# Perf summary
print("\nPERF: slowest tests")
print("-" * 70)
for p in sorted(perf_timings, key=lambda x: -x["elapsed_seconds"])[:5]:
    print(f"  {p['elapsed_seconds']:>7.2f}s  + {p['result_count']:>3} results  "
          f"[{p['test_from_state']}] {p['test_name']}")
print(f"  Total: {elapsed:.2f}s across {len(perf_timings)} tests")